Cell 1 — confirm both T4s are visible


In [1]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch; print("torch sees", torch.cuda.device_count(), "GPUs")

index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
torch sees 2 GPUs


Cell 2 — clone the repo


In [2]:
%%bash
set -e
cd /kaggle/working
if [ ! -d FreeFine ]; then
  git clone --depth 1 https://github.com/CIawevy/FreeFine.git
fi
ls FreeFine

app.py
assets
depth_anything
evaluation
Examples
generative-models
jupyter_demo
LICENSE
mystyle.css
README.md
requirements.txt
sam
scripts
src
style.css
torchhub


Cloning into 'FreeFine'...


Cell 3 — install the FreeFine inference env (system pip)


In [3]:
%%bash
set -e

# 1) uv + Python 3.10.13.
pip install -q --root-user-action=ignore uv
uv python install 3.10.13

# 2) Fresh venv at an absolute path.
VENV=/kaggle/working/freefine_env
PY=$VENV/bin/python
rm -rf "$VENV"
uv venv --python 3.10.13 "$VENV"
"$PY" -c "import sys; print('venv python:', sys.version.split()[0], sys.executable)"

# 3) Install torch first — pin --python so uv targets the venv, NOT system /usr.
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" \
    --index-url https://download.pytorch.org/whl/cu121

# 4) FreeFine's pins.
cd /kaggle/working/FreeFine
uv pip install --python "$PY" -r requirements.txt || {
  echo "Retrying with xformers unpinned..."
  grep -v '^xformers' requirements.txt > /tmp/req_noxf.txt
  uv pip install --python "$PY" -r /tmp/req_noxf.txt
  uv pip install --python "$PY" xformers
}

# 5) Extras the scripts import but aren't in requirements.txt.
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0

# 6) Sanity check (call the venv's python directly).
"$PY" - <<'PY'
import sys, torch, diffusers, xformers, transformers
print("python", sys.version.split()[0], sys.executable)
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)
print("diffusers", diffusers.__version__)
print("xformers", xformers.__version__)
print("transformers", transformers.__version__)
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 64.9 MB/s eta 0:00:00
venv python: 3.10.13 /kaggle/working/freefine_env/bin/python
python 3.10.13 /kaggle/working/freefine_env/bin/python
torch 2.1.1+cu121 cuda True 12.1
diffusers 0.18.0
xformers 0.0.23
transformers 4.30.1


Installed Python 3.10.13 in 1.43s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: freefine_env
Activate with: source freefine_env/bin/activate
Using Python 3.10.13 environment at: freefine_env
Resolved 18 packages in 1.16s
Prepared 18 packages in 26.17s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 4.23s
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/working/freefine_env
Re

Cell 4 — log in to HuggingFace and download GeoBenchMeta


In [4]:
# === Cell 4 — Download GeoBenchMeta (2D subset only) ===
import os, glob, time
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# 1) Token from Kaggle Secrets (5000 req/hr authenticated quota).
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# 2) Force the polite downloader. hf_transfer is OFF — its aggressive
#    Rust pipeline is what got us rate-limited last time.
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

DATA_DIR = "/kaggle/temp/GeoBenchMeta"
os.makedirs(DATA_DIR, exist_ok=True)

# 3) Wipe stale partial markers from earlier aria2/hf_transfer runs.
for pat in ("**/*.aria2", "**/*.incomplete"):
    for f in glob.glob(os.path.join(DATA_DIR, pat), recursive=True):
        try: os.remove(f)
        except FileNotFoundError: pass

print(f"[{time.strftime('%H:%M:%S')}] starting snapshot_download (2D subset, polite)…")

# 4) Polite resumable download. Already-present files are skipped.
snapshot_download(
    repo_id="CIawevy/GeoBenchMeta",
    repo_type="dataset",
    local_dir=DATA_DIR,
    token=os.environ["HF_TOKEN"],
    max_workers=2,           # 2 concurrent files — well under quota
    etag_timeout=60,
    allow_patterns=["annotation_2d.json", "Geo-Bench-2D/**"],
)

# 5) Verify and summarize.
print(f"[{time.strftime('%H:%M:%S')}] done. summary:")
!du -sh /kaggle/temp/GeoBenchMeta
!ls    /kaggle/temp/GeoBenchMeta
!ls    /kaggle/temp/GeoBenchMeta/Geo-Bench-2D
incomplete = !find /kaggle/temp/GeoBenchMeta -name '*.incomplete' 2>/dev/null
print("incomplete files:", len(incomplete))

[11:08:45] starting snapshot_download (2D subset, polite)…


Fetching ... files: 0it [00:00, ?it/s]

[11:44:46] done. summary:
3.0G	/kaggle/temp/GeoBenchMeta
annotation_2d.json  Geo-Bench-2D
coarse_img    source_img	  source_mask
inp_mask_vis  source_img_full_v2  target_mask
incomplete files: 0


Cell 5 — download SD-1.5 and Depth-Anything checkpoints


In [5]:
import os
from huggingface_hub import snapshot_download, hf_hub_download

CKPT_ROOT = "/kaggle/working/FreeFine/checkpoints"
SD_DIR    = f"{CKPT_ROOT}/sd-15"
DA_DIR    = f"{CKPT_ROOT}/depth-anything"
os.makedirs(DA_DIR, exist_ok=True)

# SD-1.5 (community-maintained mirror, ~5 GB).
snapshot_download(
    repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5",
    local_dir=SD_DIR,
    local_dir_use_symlinks=False,
    token=os.environ["HF_TOKEN"],
    allow_patterns=["*.json", "*.txt", "*.bin", "tokenizer/*", "scheduler/*",
                    "text_encoder/*.bin", "unet/*.bin", "vae/*.bin",
                    "feature_extractor/*", "safety_checker/*.bin"],
    max_workers=4,
)

# Depth-Anything ViT-L (only needed if you'll run the 3D pipeline).
hf_hub_download(
    repo_id="spaces/LiheYoung/Depth-Anything",
    filename="checkpoints/depth_anything_vitl14.pth",
    local_dir=DA_DIR,
    local_dir_use_symlinks=False,
    token=os.environ["HF_TOKEN"],
)
!ls -lh $SD_DIR $DA_DIR

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'spaces/LiheYoung/Depth-Anything'. Use `repo_type` argument if needed.

In [6]:
import os
from huggingface_hub import hf_hub_download

if "HF_TOKEN" not in os.environ:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

DA_DIR = "/kaggle/working/FreeFine/checkpoints/depth-anything"
os.makedirs(DA_DIR, exist_ok=True)

hf_hub_download(
    repo_id="LiheYoung/Depth-Anything",
    repo_type="space",                                # <-- the missing arg
    filename="checkpoints/depth_anything_vitl14.pth",
    local_dir=DA_DIR,
    token=os.environ["HF_TOKEN"],
)
!ls -lh /kaggle/working/FreeFine/checkpoints/depth-anything/checkpoints

checkpoints/depth_anything_vitl14.pth:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

total 1.3G
-rw-r--r-- 1 root root 1.3G May 20 11:47 depth_anything_vitl14.pth


Cell 6 — patch all hardcoded paths in their scripts


In [7]:
import re, pathlib

REPO        = "/kaggle/working/FreeFine"
GEOBENCH    = "/kaggle/temp/GeoBenchMeta"
SD15        = f"{REPO}/checkpoints/sd-15"
DEPTH_CKPT  = f"{REPO}/checkpoints/depth-anything/checkpoints/depth_anything_vitl14.pth"

EDITS = {
    f"{REPO}/evaluation/FreeFine/freefine_batch_infer_2d.py": [
        (r"sys\.path\.append\('/data/Hszhu/FreeFine'\)",
         f"sys.path.append('{REPO}')"),
        (r"/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/?",
         f"{SD15}/"),
        (r'base_dir\s*=\s*"/data/Hszhu/dataset/GeoBenchMeta/?"',
         f'base_dir = "{GEOBENCH}/"'),
    ],
    f"{REPO}/evaluation/FreeFine/freefine_batch_infer_bggen_2d.py": [
        (r"sys\.path\.append\('/data/Hszhu/FreeFine'\)",
         f"sys.path.append('{REPO}')"),
        (r"/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/?",
         f"{SD15}/"),
        (r'base_dir\s*=\s*"/data/Hszhu/dataset/GeoBenchMeta/?"',
         f'base_dir = "{GEOBENCH}/"'),
    ],
    f"{REPO}/evaluation/FreeFine/freefine_batch_infer_3d_depth.py": [
        (r"sys\.path\.append\('/data/Hszhu/FreeFine'\)",
         f"sys.path.append('{REPO}')"),
        (r"/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/?",
         f"{SD15}/"),
        (r'base_dir\s*=\s*"/data/Hszhu/dataset/GeoBenchMeta/?"',
         f'base_dir = "{GEOBENCH}/"'),
    ],
    f"{REPO}/evaluation/FreeFine/freefine_batch_infer_bggen_3d.py": [
        (r"sys\.path\.append\('/data/Hszhu/FreeFine'\)",
         f"sys.path.append('{REPO}')"),
        (r"/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/?",
         f"{SD15}/"),
        (r'base_dir\s*=\s*"/data/Hszhu/dataset/GeoBenchMeta/?"',
         f'base_dir = "{GEOBENCH}/"'),
    ],
}

for path, subs in EDITS.items():
    p = pathlib.Path(path)
    src = p.read_text()
    for pat, repl in subs:
        new = re.sub(pat, repl, src)
        assert new != src, f"Pattern not matched in {path}: {pat}"
        src = new
    p.write_text(src)
    print("patched", path)

patched /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py
patched /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_bggen_2d.py
patched /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_3d_depth.py
patched /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_bggen_3d.py


Cell 7 — 2D background inpaint (now 2-way)


In [8]:
%%bash
uv pip install --python /kaggle/working/freefine_env/bin/python "setuptools<70"
/kaggle/working/freefine_env/bin/python -c "import pkg_resources, pytorch_lightning; print('ok:', pytorch_lightning.__version__, 'setuptools:', __import__('setuptools').__version__)"

ok: 2.1.3 setuptools: 69.5.1


Using Python 3.10.13 environment at: freefine_env
Resolved 1 package in 162ms
Prepared 1 package in 86ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 32ms
 + setuptools==69.5.1
<string>:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html


In [9]:
%%bash
ls /kaggle/temp/GeoBenchMeta/annotation*.json
ln -sf /kaggle/temp/GeoBenchMeta/annotation_2d.json /kaggle/temp/GeoBenchMeta/annotations_2d.json
ls -l /kaggle/temp/GeoBenchMeta/annotation*.json

/kaggle/temp/GeoBenchMeta/annotation_2d.json
-rw-r--r-- 1 root root 5217521 May 20 11:44 /kaggle/temp/GeoBenchMeta/annotation_2d.json
lrwxrwxrwx 1 root root      44 May 20 11:48 /kaggle/temp/GeoBenchMeta/annotations_2d.json -> /kaggle/temp/GeoBenchMeta/annotation_2d.json


In [10]:
import pathlib

p = pathlib.Path("/kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_bggen_2d.py")
src = p.read_text()

# Drop the two kwargs that the bg-gen method doesn't accept.
patched = (src
    .replace('            "use_auto_draw" : False,\n', '')
    .replace('            "reduce_inp_artifacts" : True,\n', '')
)
assert patched != src, "no replacement happened — open the file and check the indentation"
p.write_text(patched)
print("patched:", p)

# Verify the params dict now matches FreeFine_background_generation's signature.
import re
m = re.search(r'params = \{.*?\}', patched, flags=re.S)
print(m.group(0))

patched: /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_bggen_2d.py
params = {
            "ori_img": ori_img,
            "ori_mask": ori_mask,
            "guidance_text": "empty scene",
            "guidance_scale": 7.5,
            "eta": 1.0,
            "end_scale": 0.5,
            "end_step": 35,
            "num_step": 50,
            "start_step": 1,
            "seed": seed_r,
            "return_intermediates" : False,
        }


Patch — chunk mask_attention

In [11]:
import pathlib

p = pathlib.Path("/kaggle/working/FreeFine/src/utils/attention.py")
src = p.read_text()

old = '''def mask_attention(self,query,key,value,attention_mask):
        attention_probs = self.get_attention_scores(query, key, attention_mask)
        return torch.bmm(attention_probs, value)'''

new = '''def mask_attention(self,query,key,value,attention_mask):
        # Chunked over batch*head dim — same math, lower peak memory.
        import os as _os
        dtype = query.dtype
        N = query.shape[0]
        chunk = int(_os.environ.get("FREEFINE_ATTN_CHUNK", "1"))
        outs = []
        for i in range(0, N, chunk):
            q_i = query[i:i+chunk]
            k_i = key[i:i+chunk]
            v_i = value[i:i+chunk]
            m_i = None if attention_mask is None else attention_mask[i:i+chunk]
            if getattr(self, "upcast_attention", False):
                q_i = q_i.float(); k_i = k_i.float()
            if m_i is None:
                baddbmm_input = torch.empty(q_i.shape[0], q_i.shape[1], k_i.shape[1],
                                            dtype=q_i.dtype, device=q_i.device)
                beta = 0
            else:
                baddbmm_input = m_i
                beta = 1
            scores = torch.baddbmm(baddbmm_input, q_i, k_i.transpose(-1,-2),
                                   beta=beta, alpha=self.scale)
            del baddbmm_input
            if getattr(self, "upcast_softmax", False):
                scores = scores.float()
            probs = scores.softmax(dim=-1).to(dtype)
            del scores
            outs.append(torch.bmm(probs, v_i))
            del probs
        return torch.cat(outs, dim=0)'''

assert old in src, "could not locate original mask_attention block"
p.write_text(src.replace(old, new))
print("patched src/utils/attention.py — mask_attention is now chunked")

patched src/utils/attention.py — mask_attention is now chunked


In [12]:
import subprocess, time, glob, os, signal, socket, datetime

VENV   = "/kaggle/working/freefine_env"
SCRIPT = "/kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_bggen_2d.py"
OUTDIR = "/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended"
LOG    = "/kaggle/working/bggen_2d.log"
TOTAL  = 668

env = os.environ.copy()
env.update({
    "PATH": f"{VENV}/bin:" + env.get("PATH", ""),
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "2",
    "NCCL_P2P_DISABLE": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "FREEFINE_ATTN_CHUNK": "8",
})

# pick a free TCP port for torchrun rendezvous
s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
cmd = [f"{VENV}/bin/torchrun", "--nproc_per_node=2", f"--master-port={port}", SCRIPT]

print(f"[start] {' '.join(cmd)}", flush=True)
print(f"[start] cwd=/kaggle/temp/GeoBenchMeta  ATTN_CHUNK=4  log={LOG}", flush=True)

log_f = open(LOG, "w")
proc = subprocess.Popen(
    cmd,
    cwd="/kaggle/temp/GeoBenchMeta",
    env=env,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,            # own process group → interrupt kills children cleanly
)
print(f"[start] pid={proc.pid} pgid={os.getpgid(proc.pid)}", flush=True)

start = time.time()
last_report = 0.0
REPORT_EVERY = 30                    # seconds between progress prints

def count_done():
    return len(glob.glob(f"{OUTDIR}/*/*/inp_img.png"))

try:
    while proc.poll() is None:
        time.sleep(2)
        now = time.time()
        if now - last_report >= REPORT_EVERY:
            count = count_done()
            elapsed_min = (now - start) / 60
            ts = datetime.datetime.now().strftime("%H:%M:%S")
            if count > 0 and elapsed_min > 0:
                rate = count / elapsed_min
                eta = (TOTAL - count) / rate if rate > 0 else float("inf")
                print(f"[{ts}] done={count}/{TOTAL}  elapsed={elapsed_min:.1f}min  "
                      f"rate={rate:.1f}/min  ETA={eta:.0f}min", flush=True)
            else:
                print(f"[{ts}] done={count}/{TOTAL}  (still loading models / first cases)", flush=True)
            last_report = now
except KeyboardInterrupt:
    print("[interrupt] sending SIGINT to process group...", flush=True)
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGINT)
        proc.wait(timeout=30)
    except Exception as e:
        print(f"[interrupt] cleanup error: {e}", flush=True)
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:
            pass

log_f.close()
count = count_done()
print(f"[done] rc={proc.returncode}  final={count}/{TOTAL}  log={LOG}", flush=True)

[start] /kaggle/working/freefine_env/bin/torchrun --nproc_per_node=2 --master-port=50131 /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_bggen_2d.py
[start] cwd=/kaggle/temp/GeoBenchMeta  ATTN_CHUNK=4  log=/kaggle/working/bggen_2d.log
[start] pid=17309 pgid=17309
[11:48:28] done=0/668  (still loading models / first cases)
[11:48:58] done=0/668  (still loading models / first cases)
[11:49:28] done=0/668  (still loading models / first cases)
[11:49:58] done=0/668  (still loading models / first cases)
[11:50:28] done=0/668  (still loading models / first cases)
[11:50:58] done=0/668  (still loading models / first cases)
[11:51:28] done=2/668  elapsed=3.0min  rate=0.7/min  ETA=1010min
[11:51:58] done=2/668  elapsed=3.5min  rate=0.6/min  ETA=1177min
[11:52:28] done=2/668  elapsed=4.0min  rate=0.5/min  ETA=1344min
[11:52:58] done=3/668  elapsed=4.5min  rate=0.7/min  ETA=1005min
[11:53:28] done=4/668  elapsed=5.0min  rate=0.8/min  ETA=836min
[11:53:58] done=4/668  elapsed=5.5

Resume the missing 11 — single-rank, no NCCL

In [15]:
import subprocess, time, glob, os, signal, socket, datetime

VENV   = "/kaggle/working/freefine_env"
SCRIPT = "/kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_bggen_2d.py"
OUTDIR = "/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended"
LOG    = "/kaggle/working/bggen_2d_tail.log"
TOTAL  = 668

env = os.environ.copy()
env.update({
    "PATH": f"{VENV}/bin:" + env.get("PATH", ""),
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "2",
    "NCCL_P2P_DISABLE": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "FREEFINE_ATTN_CHUNK": "8",
})

s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
cmd = [f"{VENV}/bin/torchrun", "--nproc_per_node=1", f"--master-port={port}", SCRIPT]

print(f"[start] single-rank cleanup", flush=True)
log_f = open(LOG, "w")
proc = subprocess.Popen(
    cmd, cwd="/kaggle/temp/GeoBenchMeta", env=env,
    stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid,
)
print(f"[start] pid={proc.pid}", flush=True)

start = time.time(); last_report = 0.0
try:
    while proc.poll() is None:
        time.sleep(2)
        now = time.time()
        if now - last_report >= 30:
            count = len(glob.glob(f"{OUTDIR}/*/*/inp_img.png"))
            ts = datetime.datetime.now().strftime("%H:%M:%S")
            print(f"[{ts}] done={count}/{TOTAL}  elapsed={(now-start)/60:.1f}min", flush=True)
            last_report = now
except KeyboardInterrupt:
    print("[interrupt] killing pgid", flush=True)
    try: os.killpg(os.getpgid(proc.pid), signal.SIGINT); proc.wait(timeout=30)
    except Exception: pass

log_f.close()
count = len(glob.glob(f"{OUTDIR}/*/*/inp_img.png"))
print(f"[done] rc={proc.returncode}  final={count}/{TOTAL}", flush=True)

[start] single-rank cleanup
[start] pid=17387
[20:53:31] done=657/668  elapsed=0.0min
[20:54:01] done=657/668  elapsed=0.5min
[20:54:31] done=657/668  elapsed=1.0min
[20:55:01] done=657/668  elapsed=1.5min
[20:55:31] done=658/668  elapsed=2.0min
[20:56:02] done=658/668  elapsed=2.5min
[20:56:32] done=658/668  elapsed=3.0min
[20:57:02] done=659/668  elapsed=3.5min
[20:57:32] done=659/668  elapsed=4.0min
[20:58:02] done=659/668  elapsed=4.5min
[20:58:32] done=660/668  elapsed=5.0min
[20:59:02] done=660/668  elapsed=5.5min
[20:59:32] done=660/668  elapsed=6.0min
[21:00:02] done=660/668  elapsed=6.5min
[21:00:32] done=661/668  elapsed=7.0min
[21:01:02] done=661/668  elapsed=7.5min
[21:01:32] done=661/668  elapsed=8.0min
[21:02:02] done=662/668  elapsed=8.5min
[21:02:32] done=662/668  elapsed=9.0min
[21:03:02] done=662/668  elapsed=9.5min
[21:03:32] done=663/668  elapsed=10.0min
[21:04:02] done=663/668  elapsed=10.5min
[21:04:32] done=663/668  elapsed=11.0min
[21:05:02] done=664/668  elapse

Cell A — back up Cell 7's output to /kaggle/working

In [16]:
%%bash
set -e
SRC=/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended
DST=/kaggle/working/inp_img_blended.tar
echo "tarring $(find $SRC -name inp_img.png | wc -l) files..."
tar -cf $DST -C /kaggle/temp/GeoBenchMeta/Geo-Bench-2D inp_img_blended
ls -lh $DST
df -h /kaggle/working

tarring 668 files...
-rw-r--r-- 1 root root 220M May 20 21:17 /kaggle/working/inp_img_blended.tar
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   19G  1.3G  94% /kaggle/working


In [17]:
%%bash
ls -lh /kaggle/working/inp_img_blended.tar
tar -tf /kaggle/working/inp_img_blended.tar | wc -l

-rw-r--r-- 1 root root 220M May 20 21:17 /kaggle/working/inp_img_blended.tar
1946


---- Here the session was ended aprox 11h in, and started a new one, so below are the cells nedded for it.

Cell N1 — HF_TOKEN (1 sec)

In [1]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN set, length:", len(os.environ["HF_TOKEN"]))

HF_TOKEN set, length: 37


Cell N2 — download GeoBenchMeta 2D subset (~15 min)



In [2]:
import os
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="CIawevy/GeoBenchMeta",
    repo_type="dataset",
    local_dir="/kaggle/temp/GeoBenchMeta",
    token=os.environ["HF_TOKEN"],
    max_workers=2,
    etag_timeout=60,
    allow_patterns=["annotation_2d.json", "Geo-Bench-2D/**"],
)
!du -sh /kaggle/temp/GeoBenchMeta && ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

Fetching ... files: 0it [00:00, ?it/s]

3.0G	/kaggle/temp/GeoBenchMeta
coarse_img    source_img	  source_mask
inp_mask_vis  source_img_full_v2  target_mask


Cell N3 — annotation_2d.json symlink (instant)



In [3]:
%%bash
ln -sf /kaggle/temp/GeoBenchMeta/annotation_2d.json /kaggle/temp/GeoBenchMeta/annotations_2d.json
ls -l /kaggle/temp/GeoBenchMeta/annotation*.json

-rw-r--r-- 1 root root 5217521 May 20 22:08 /kaggle/temp/GeoBenchMeta/annotation_2d.json
lrwxrwxrwx 1 root root      44 May 20 22:08 /kaggle/temp/GeoBenchMeta/annotations_2d.json -> /kaggle/temp/GeoBenchMeta/annotation_2d.json


Cell N4 — restore Cell 7's outputs from your tar (~30 sec)



In [4]:
%%bash
set -e
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D
tar -xf /kaggle/working/inp_img_blended.tar -C /kaggle/temp/GeoBenchMeta/Geo-Bench-2D
echo "restored — inp_img.png count (must be 668):"
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended -name inp_img.png | wc -l

restored — inp_img.png count (must be 668):
668


Cell 8 — 2D edit inference (2-way)


Fix — restore exec bits on the venv

In [6]:
%%bash
chmod +x /kaggle/working/freefine_env/bin/*
ls -l /kaggle/working/freefine_env/bin/torchrun /kaggle/working/freefine_env/bin/python
/kaggle/working/freefine_env/bin/python --version

-rwxr-xr-x 1 root root 326 May 20 21:37 /kaggle/working/freefine_env/bin/torchrun


ls: cannot access '/kaggle/working/freefine_env/bin/python': No such file or directory
bash: line 3: /kaggle/working/freefine_env/bin/python: No such file or directory


CalledProcessError: Command 'b'chmod +x /kaggle/working/freefine_env/bin/*\nls -l /kaggle/working/freefine_env/bin/torchrun /kaggle/working/freefine_env/bin/python\n/kaggle/working/freefine_env/bin/python --version\n'' returned non-zero exit status 127.

Rebuild venv (replaces Cell 3 for this session)
This venv-rebuild step needs to run on every new session — uv's Python interpreter doesn't survive Kaggle's persistence reset. 

In [7]:
%%bash
set -e

pip install -q --root-user-action=ignore uv
uv python install 3.10.13

VENV=/kaggle/working/freefine_env
PY=$VENV/bin/python
rm -rf "$VENV"
uv venv --python 3.10.13 "$VENV"
"$PY" -c "import sys; print('venv:', sys.version.split()[0], sys.executable)"

uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" \
    --index-url https://download.pytorch.org/whl/cu121

cd /kaggle/working/FreeFine
uv pip install --python "$PY" -r requirements.txt || {
  echo "Retrying with xformers unpinned..."
  grep -v '^xformers' requirements.txt > /tmp/req_noxf.txt
  uv pip install --python "$PY" -r /tmp/req_noxf.txt
  uv pip install --python "$PY" xformers
}

# Extras + the setuptools<70 fix for pkg_resources
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70" wheel pip

"$PY" - <<'PY'
import sys, torch, diffusers, xformers, transformers, pkg_resources, pytorch_lightning
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)
print("diffusers", diffusers.__version__)
print("xformers", xformers.__version__)
print("transformers", transformers.__version__)
print("pytorch_lightning", pytorch_lightning.__version__)
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 64.7 MB/s eta 0:00:00
venv: 3.10.13 /kaggle/working/freefine_env/bin/python
python 3.10.13
torch 2.1.1+cu121 cuda True 12.1
diffusers 0.18.0
xformers 0.0.23
transformers 4.30.1
pytorch_lightning 2.1.3


Installed Python 3.10.13 in 1.35s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: freefine_env
Activate with: source freefine_env/bin/activate
Using Python 3.10.13 environment at: freefine_env
Resolved 18 packages in 1.19s
Prepared 18 packages in 31.25s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 5.15s
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/working/freefine_env
Re

Fix the variable-order bug


In [11]:
import pathlib

p = pathlib.Path("/kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py")
src = p.read_text()

old = '''        ori_img = read_and_resize_img(ori_img_path)
        # target_mask = read_and_resize_mask(tgt_mask_path)
        # coarse_input = read_and_resize_img(coarse_input_path)

        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)
        obj_label = ""
        ori_mask = read_and_resize_mask(ori_mask_path)'''

new = '''        ori_img = read_and_resize_img(ori_img_path)
        ori_mask = read_and_resize_mask(ori_mask_path)
        # target_mask = read_and_resize_mask(tgt_mask_path)
        # coarse_input = read_and_resize_img(coarse_input_path)

        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)
        obj_label = ""'''

assert old in src, "couldn't locate the buggy block"
p.write_text(src.replace(old, new))
print("patched freefine_batch_infer_2d.py — ori_mask now loaded before re_edit_2d")

patched freefine_batch_infer_2d.py — ori_mask now loaded before re_edit_2d


In [ ]:
import subprocess, time, os, signal, socket, datetime

VENV   = "/kaggle/working/freefine_env"
SCRIPT = "/kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py"
OUTDIR = "/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d"
BACKUP = "/kaggle/working/gen_results_2d_backup"
LOG    = "/kaggle/working/edit_2d.log"

os.makedirs(BACKUP, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

env = os.environ.copy()
env.update({
    "PATH": f"{VENV}/bin:" + env.get("PATH", ""),
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "2",
    "NCCL_P2P_DISABLE": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "FREEFINE_ATTN_CHUNK": "8",          # fp32, attention NOT chunked
})

s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
cmd = [f"{VENV}/bin/torchrun", "--nproc_per_node=2", f"--master-port={port}", SCRIPT]

print(f"[start] Cell 8 — fp32 edit step, 2 ranks", flush=True)
print(f"[start] script={os.path.basename(SCRIPT)}  log={LOG}", flush=True)
print(f"[start] output={OUTDIR}", flush=True)
print(f"[start] backup={BACKUP} (every 10 min)", flush=True)

log_f = open(LOG, "w")
proc = subprocess.Popen(
    cmd, cwd="/kaggle/temp/GeoBenchMeta", env=env,
    stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid,
)
print(f"[start] pid={proc.pid}", flush=True)

def count_done():
    if not os.path.isdir(OUTDIR): return 0
    n = 0
    for _, _, files in os.walk(OUTDIR):
        n += sum(1 for f in files if f.endswith('.png'))
    return n

def backup_outputs():
    if os.path.isdir(OUTDIR):
        subprocess.run(["rsync", "-a", OUTDIR + "/", BACKUP + "/"],
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

start = time.time()
last_report = 0.0
last_backup = 0.0
REPORT_EVERY = 30
BACKUP_EVERY = 600

try:
    while proc.poll() is None:
        time.sleep(2)
        now = time.time()
        if now - last_report >= REPORT_EVERY:
            count = count_done()
            ts = datetime.datetime.now().strftime("%H:%M:%S")
            elapsed_min = (now - start) / 60
            if count > 0 and elapsed_min > 0:
                rate = count / elapsed_min
                print(f"[{ts}] generated={count}  elapsed={elapsed_min:.1f}min  "
                      f"rate={rate:.1f}/min", flush=True)
            else:
                print(f"[{ts}] generated={count}  (loading / first cases)", flush=True)
            last_report = now
        if now - last_backup >= BACKUP_EVERY:
            backup_outputs()
            print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] backup → {BACKUP}", flush=True)
            last_backup = now
except KeyboardInterrupt:
    print("[interrupt] killing pgid...", flush=True)
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGINT)
        proc.wait(timeout=30)
    except Exception:
        try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception: pass

backup_outputs()
log_f.close()
count = count_done()
print(f"[done] rc={proc.returncode}  generated={count}  log={LOG}", flush=True)
print(f"[done] outputs at {OUTDIR}", flush=True)
print(f"[done] backup at  {BACKUP}", flush=True)

[start] Cell 8 — fp32 edit step, 2 ranks
[start] script=freefine_batch_infer_2d.py  log=/kaggle/working/edit_2d.log
[start] output=/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d
[start] backup=/kaggle/working/gen_results_2d_backup (every 10 min)
[start] pid=16293
[22:30:21] generated=0  (loading / first cases)
[22:30:21] backup → /kaggle/working/gen_results_2d_backup
[22:30:51] generated=0  (loading / first cases)
[22:31:21] generated=2  elapsed=1.0min  rate=1.9/min
[22:31:51] generated=4  elapsed=1.5min  rate=2.6/min
[22:32:21] generated=4  elapsed=2.0min  rate=2.0/min
[22:32:52] generated=6  elapsed=2.5min  rate=2.4/min
[22:33:22] generated=8  elapsed=3.0min  rate=2.6/min
[22:33:52] generated=9  elapsed=3.5min  rate=2.5/min
[22:34:22] generated=10  elapsed=4.0min  rate=2.5/min
[22:34:52] generated=12  elapsed=4.5min  rate=2.6/min
[22:35:22] generated=13  elapsed=5.0min  rate=2.6/min
[22:35:52] generated=15  elapsed=5.5min  rate=2.7/min
[22:36:22] generated=16  elapsed

New session:
Step 0 — verify what survived persistence


In [1]:
%%bash
echo "=== /kaggle/working state ==="
ls -lh /kaggle/working/*.tar 2>/dev/null
echo
echo "=== FreeFine repo + ckpts? ==="
ls /kaggle/working/FreeFine/checkpoints/sd-15/unet/ 2>/dev/null | head -3
grep -c "ori_mask = read_and_resize_mask" /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py
echo
echo "=== free disk ==="
df -h /kaggle/working

=== /kaggle/working state ===
-rw-r--r-- 1 root root 749M May 23 09:56 /kaggle/working/gen_results_2d_partial.tar
-rw-r--r-- 1 root root 220M May 23 09:56 /kaggle/working/inp_img_blended.tar

=== FreeFine repo + ckpts? ===
config.json
diffusion_pytorch_model.bin
diffusion_pytorch_model.fp16.bin
1

=== free disk ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  5.5G  72% /kaggle/working


Step 1 — set HF_TOKEN


In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN set, length:", len(os.environ["HF_TOKEN"]))

HF_TOKEN set, length: 37


Step 2 — re-download GeoBenchMeta 2D subset (~15 min)


In [3]:
import os
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="CIawevy/GeoBenchMeta",
    repo_type="dataset",
    local_dir="/kaggle/temp/GeoBenchMeta",
    token=os.environ["HF_TOKEN"],
    max_workers=2,
    etag_timeout=60,
    allow_patterns=["annotation_2d.json", "Geo-Bench-2D/**"],
)
!du -sh /kaggle/temp/GeoBenchMeta && ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

Fetching ... files: 0it [00:00, ?it/s]

3.0G	/kaggle/temp/GeoBenchMeta
coarse_img    source_img	  source_mask
inp_mask_vis  source_img_full_v2  target_mask


Step 3 — symlink annotations + restore both tars


In [4]:
%%bash
set -e
# Symlink for the misnamed annotations file
ln -sf /kaggle/temp/GeoBenchMeta/annotation_2d.json /kaggle/temp/GeoBenchMeta/annotations_2d.json

# Restore Cell 7's 668 bg-gen outputs into the live data tree
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D
tar -xf /kaggle/working/inp_img_blended.tar -C /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

# Restore the partial Cell 8 outputs into the live Gen_results dir
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d
tar -xf /kaggle/working/gen_results_2d_partial.tar -C /kaggle/working
rsync -a /kaggle/working/gen_results_2d_backup/ \
         /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/

echo "counts (must be 668 and 1924):"
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended -name 'inp_img.png' | wc -l
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' | wc -l

counts (must be 668 and 1924):
668
1924


Step 4 — rebuild the venv (~5 min)


In [7]:
%%bash
set -e

pip install -q --root-user-action=ignore uv
uv python install 3.10.13

VENV=/kaggle/working/freefine_env
PY=$VENV/bin/python
rm -rf "$VENV"
uv venv --python 3.10.13 "$VENV"
"$PY" -c "import sys; print('venv:', sys.version.split()[0], sys.executable)"

uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" \
    --index-url https://download.pytorch.org/whl/cu121

cd /kaggle/working/FreeFine
uv pip install --python "$PY" -r requirements.txt || {
  echo "Retrying with xformers unpinned..."
  grep -v '^xformers' requirements.txt > /tmp/req_noxf.txt
  uv pip install --python "$PY" -r /tmp/req_noxf.txt
  uv pip install --python "$PY" xformers
}

uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70" wheel pip

"$PY" - <<'PY'
import sys, torch, diffusers, xformers, transformers, pkg_resources, pytorch_lightning
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)
print("diffusers", diffusers.__version__)
print("xformers", xformers.__version__)
print("transformers", transformers.__version__)
print("pytorch_lightning", pytorch_lightning.__version__)
PY

venv: 3.10.13 /kaggle/working/freefine_env/bin/python
python 3.10.13
torch 2.1.1+cu121 cuda True 12.1
diffusers 0.18.0
xformers 0.0.23
transformers 4.30.1
pytorch_lightning 2.1.3


Python 3.10.13 is already installed
Using CPython 3.10.13
Creating virtual environment at: freefine_env
Activate with: source freefine_env/bin/activate
Using Python 3.10.13 environment at: freefine_env
Resolved 18 packages in 529ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 21.35s
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/working/freefine_env
Resolved 186 packages in 2.07s
Uninstalled 6 packages in 48ms
         If the c

Disk full. Cleanup cell
 The current venv is broken. Fix in two cells: free space (delete venv + redundant SD-1.5 weights + the duplicated backup folder), then re-run Step 4 cleanly.

In [6]:
%%bash
set -e

# 1) Remove the failed/partial venv (~8 GB).
rm -rf /kaggle/working/freefine_env

# 2) Remove the rsync'd copy of the partial outputs — we still have:
#    - the live copy in /kaggle/temp/.../Gen_results_FreeFine_2d/
#    - the tar in /kaggle/working/gen_results_2d_partial.tar
#    - the upload in your Kaggle Dataset
# The supervisor will recreate this folder automatically.
rm -rf /kaggle/working/gen_results_2d_backup

# 3) Remove redundant SD-1.5 weight variants we never load (fp16 + non_ema in unet).
#    Pipeline uses diffusion_pytorch_model.bin (fp32 EMA) by default.
ls -lh /kaggle/working/FreeFine/checkpoints/sd-15/unet/
rm -f /kaggle/working/FreeFine/checkpoints/sd-15/unet/diffusion_pytorch_model.fp16.bin
rm -f /kaggle/working/FreeFine/checkpoints/sd-15/unet/diffusion_pytorch_model.non_ema.bin

# 4) Remove other component fp16 variants if present (small savings, no harm).
find /kaggle/working/FreeFine/checkpoints/sd-15 -name '*.fp16.bin' -delete 2>/dev/null
find /kaggle/working/FreeFine/checkpoints/sd-15 -name '*.fp16.safetensors' -delete 2>/dev/null

echo "after cleanup:"
df -h /kaggle/working
du -sh /kaggle/working/FreeFine/checkpoints/sd-15

total 8.1G
-rw-r--r-- 1 root root  743 May 23 09:57 config.json
-rw-r--r-- 1 root root 3.3G May 23 09:57 diffusion_pytorch_model.bin
-rw-r--r-- 1 root root 1.7G May 23 09:57 diffusion_pytorch_model.fp16.bin
-rw-r--r-- 1 root root 3.3G May 23 09:57 diffusion_pytorch_model.non_ema.bin
after cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  7.6G   12G  39% /kaggle/working
5.2G	/kaggle/working/FreeFine/checkpoints/sd-15


Step 5 — Cell 8 supervisor (resume; will skip the 1924 already done)


In [8]:
import subprocess, time, os, signal, socket, datetime

VENV   = "/kaggle/working/freefine_env"
SCRIPT = "/kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py"
OUTDIR = "/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d"
BACKUP = "/kaggle/working/gen_results_2d_backup"
LOG    = "/kaggle/working/edit_2d.log"

os.makedirs(BACKUP, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

env = os.environ.copy()
env.update({
    "PATH": f"{VENV}/bin:" + env.get("PATH", ""),
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "2",
    "NCCL_P2P_DISABLE": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "FREEFINE_ATTN_CHUNK": "8",          # fp32, attention NOT chunked
})

s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
cmd = [f"{VENV}/bin/torchrun", "--nproc_per_node=2", f"--master-port={port}", SCRIPT]

print(f"[start] Cell 8 — fp32 edit step, 2 ranks  log={LOG}", flush=True)
log_f = open(LOG, "w")
proc = subprocess.Popen(
    cmd, cwd="/kaggle/temp/GeoBenchMeta", env=env,
    stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid,
)
print(f"[start] pid={proc.pid}", flush=True)

def count_done():
    if not os.path.isdir(OUTDIR): return 0
    n = 0
    for _, _, files in os.walk(OUTDIR):
        n += sum(1 for f in files if f.endswith('.png'))
    return n

def backup_outputs():
    if os.path.isdir(OUTDIR):
        subprocess.run(["rsync", "-a", OUTDIR + "/", BACKUP + "/"],
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

start = time.time(); last_report = 0.0; last_backup = 0.0
REPORT_EVERY = 30; BACKUP_EVERY = 600

try:
    while proc.poll() is None:
        time.sleep(2); now = time.time()
        if now - last_report >= REPORT_EVERY:
            count = count_done(); ts = datetime.datetime.now().strftime("%H:%M:%S")
            elapsed_min = (now - start) / 60
            if count > 0 and elapsed_min > 0:
                rate = count / elapsed_min
                print(f"[{ts}] generated={count}  elapsed={elapsed_min:.1f}min  rate={rate:.1f}/min", flush=True)
            else:
                print(f"[{ts}] generated={count}  (loading / first cases)", flush=True)
            last_report = now
        if now - last_backup >= BACKUP_EVERY:
            backup_outputs()
            print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] backup → {BACKUP}", flush=True)
            last_backup = now
except KeyboardInterrupt:
    print("[interrupt] killing pgid", flush=True)
    try: os.killpg(os.getpgid(proc.pid), signal.SIGINT); proc.wait(timeout=30)
    except Exception:
        try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception: pass

backup_outputs()
log_f.close()
count = count_done()
print(f"[done] rc={proc.returncode}  generated={count}/5677  log={LOG}", flush=True)

[start] Cell 8 — fp32 edit step, 2 ranks  log=/kaggle/working/edit_2d.log
[start] pid=16397
[10:38:08] generated=1924  elapsed=0.0min  rate=57664.8/min
[10:38:13] backup → /kaggle/working/gen_results_2d_backup
[10:38:39] generated=1924  elapsed=0.5min  rate=3509.0/min
[10:39:09] generated=1924  elapsed=1.0min  rate=1834.7/min
[10:39:39] generated=1924  elapsed=1.5min  rate=1242.1/min
[10:40:09] generated=1924  elapsed=2.0min  rate=938.7/min
[10:40:39] generated=1926  elapsed=2.6min  rate=755.2/min
[10:41:09] generated=1928  elapsed=3.1min  rate=631.9/min
[10:41:39] generated=1929  elapsed=3.6min  rate=543.1/min
[10:42:09] generated=1931  elapsed=4.1min  rate=476.5/min
[10:42:39] generated=1932  elapsed=4.6min  rate=424.3/min
[10:43:09] generated=1933  elapsed=5.1min  rate=382.4/min
[10:43:39] generated=1935  elapsed=5.6min  rate=348.3/min
[10:44:09] generated=1937  elapsed=6.1min  rate=319.9/min
[10:44:39] generated=1938  elapsed=6.6min  rate=295.6/min
[10:45:09] generated=1939  elapse

KeyboardInterrupt: 

Stopped the cell run to start a new session.
Step B — verify the backup is current


In [9]:
%%bash
echo "live (in /kaggle/temp):"
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' | wc -l
echo "backup (in /kaggle/working):"
find /kaggle/working/gen_results_2d_backup -name '*.png' | wc -l
df -h /kaggle/working

live (in /kaggle/temp):
3680
backup (in /kaggle/working):
3670
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  4.9G  76% /kaggle/working


In [10]:
%%bash
set -e

# 1) Force-sync any cases the supervisor's final backup missed.
rsync -a /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/ \
         /kaggle/working/gen_results_2d_backup/
echo "after sync — backup count:"
find /kaggle/working/gen_results_2d_backup -name '*.png' | wc -l

# 2) Free space for the new tar (venv rebuilds next session anyway).
rm -rf /kaggle/working/freefine_env
rm -f  /kaggle/working/gen_results_2d_partial.tar
df -h /kaggle/working

# 3) Create the new tar with all ~3680 PNGs.
TAR=/kaggle/working/gen_results_2d_partial.tar
tar -cf $TAR -C /kaggle/working gen_results_2d_backup
ls -lh $TAR
df -h /kaggle/working

after sync — backup count:
3680
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  8.2G   12G  43% /kaggle/working
-rw-r--r-- 1 root root 1.4G May 23 20:54 /kaggle/working/gen_results_2d_partial.tar
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  9.6G   10G  50% /kaggle/working


New Session!!

Step 0 — verify what survived persistence

In [1]:
%%bash
echo "=== /kaggle/working state ==="
ls -lh /kaggle/working/*.tar 2>/dev/null
echo
echo "=== FreeFine repo + ckpts? ==="
ls /kaggle/working/FreeFine/checkpoints/sd-15/unet/ 2>/dev/null | head -3
grep -c "ori_mask = read_and_resize_mask" /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py
echo
echo "=== free disk ==="
df -h /kaggle/working

=== /kaggle/working state ===
-rw-r--r-- 1 root root 1.4G May 23 21:20 /kaggle/working/gen_results_2d_partial.tar
-rw-r--r-- 1 root root 220M May 23 21:20 /kaggle/working/inp_img_blended.tar

=== FreeFine repo + ckpts? ===
config.json
diffusion_pytorch_model.bin
1

=== free disk ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  9.6G   10G  50% /kaggle/working


Step 1 — set HF_TOKEN

In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN set, length:", len(os.environ["HF_TOKEN"]))

HF_TOKEN set, length: 37


Step 2 — re-download GeoBenchMeta 2D subset (~15 min)


In [3]:
import os
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="CIawevy/GeoBenchMeta",
    repo_type="dataset",
    local_dir="/kaggle/temp/GeoBenchMeta",
    token=os.environ["HF_TOKEN"],
    max_workers=2,
    etag_timeout=60,
    allow_patterns=["annotation_2d.json", "Geo-Bench-2D/**"],
)
!du -sh /kaggle/temp/GeoBenchMeta && ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

Fetching ... files: 0it [00:00, ?it/s]

3.0G	/kaggle/temp/GeoBenchMeta
coarse_img    source_img	  source_mask
inp_mask_vis  source_img_full_v2  target_mask


Step 3 — symlink annotations + restore both tars


In [4]:
%%bash
set -e
# Symlink for the misnamed annotations file
ln -sf /kaggle/temp/GeoBenchMeta/annotation_2d.json /kaggle/temp/GeoBenchMeta/annotations_2d.json

# Restore Cell 7's 668 bg-gen outputs into the live data tree
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D
tar -xf /kaggle/working/inp_img_blended.tar -C /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

# Restore the partial Cell 8 outputs into the live Gen_results dir
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d
tar -xf /kaggle/working/gen_results_2d_partial.tar -C /kaggle/working
rsync -a /kaggle/working/gen_results_2d_backup/ \
         /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/

echo "counts (must be 668 and 1924):"
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended -name 'inp_img.png' | wc -l
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' | wc -l

counts (must be 668 and 1924):
668
3680


Step 4 — rebuild the venv (~5 min)


In [5]:
%%bash
set -e

pip install -q --root-user-action=ignore uv
uv python install 3.10.13

VENV=/kaggle/working/freefine_env
PY=$VENV/bin/python
rm -rf "$VENV"
uv venv --python 3.10.13 "$VENV"
"$PY" -c "import sys; print('venv:', sys.version.split()[0], sys.executable)"

uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" \
    --index-url https://download.pytorch.org/whl/cu121

cd /kaggle/working/FreeFine
uv pip install --python "$PY" -r requirements.txt || {
  echo "Retrying with xformers unpinned..."
  grep -v '^xformers' requirements.txt > /tmp/req_noxf.txt
  uv pip install --python "$PY" -r /tmp/req_noxf.txt
  uv pip install --python "$PY" xformers
}

uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70" wheel pip

"$PY" - <<'PY'
import sys, torch, diffusers, xformers, transformers, pkg_resources, pytorch_lightning
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)
print("diffusers", diffusers.__version__)
print("xformers", xformers.__version__)
print("transformers", transformers.__version__)
print("pytorch_lightning", pytorch_lightning.__version__)
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 61.4 MB/s eta 0:00:00
venv: 3.10.13 /kaggle/working/freefine_env/bin/python
python 3.10.13
torch 2.1.1+cu121 cuda True 12.1
diffusers 0.18.0
xformers 0.0.23
transformers 4.30.1
pytorch_lightning 2.1.3


Installed Python 3.10.13 in 1.29s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: freefine_env
Activate with: source freefine_env/bin/activate
Using Python 3.10.13 environment at: freefine_env
Resolved 18 packages in 966ms
Prepared 18 packages in 26.11s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 5.77s
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/working/freefine_env
Re

Disk full. Cleanup cell
 The current venv is broken. Fix in two cells: free space (delete venv + redundant SD-1.5 weights + the duplicated backup folder), then re-run Step 4 cleanly.

In [ ]:
%%bash
set -e

# 1) Remove the failed/partial venv (~8 GB).
rm -rf /kaggle/working/freefine_env

# 2) Remove the rsync'd copy of the partial outputs — we still have:
#    - the live copy in /kaggle/temp/.../Gen_results_FreeFine_2d/
#    - the tar in /kaggle/working/gen_results_2d_partial.tar
#    - the upload in your Kaggle Dataset
# The supervisor will recreate this folder automatically.
rm -rf /kaggle/working/gen_results_2d_backup

# 3) Remove redundant SD-1.5 weight variants we never load (fp16 + non_ema in unet).
#    Pipeline uses diffusion_pytorch_model.bin (fp32 EMA) by default.
ls -lh /kaggle/working/FreeFine/checkpoints/sd-15/unet/
rm -f /kaggle/working/FreeFine/checkpoints/sd-15/unet/diffusion_pytorch_model.fp16.bin
rm -f /kaggle/working/FreeFine/checkpoints/sd-15/unet/diffusion_pytorch_model.non_ema.bin

# 4) Remove other component fp16 variants if present (small savings, no harm).
find /kaggle/working/FreeFine/checkpoints/sd-15 -name '*.fp16.bin' -delete 2>/dev/null
find /kaggle/working/FreeFine/checkpoints/sd-15 -name '*.fp16.safetensors' -delete 2>/dev/null

echo "after cleanup:"
df -h /kaggle/working
du -sh /kaggle/working/FreeFine/checkpoints/sd-15

Step 5 — Cell 8 supervisor. First progress line should show generated=~3665 then advance.

In [6]:
import subprocess, time, os, signal, socket, datetime

VENV   = "/kaggle/working/freefine_env"
SCRIPT = "/kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py"
OUTDIR = "/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d"
BACKUP = "/kaggle/working/gen_results_2d_backup"
LOG    = "/kaggle/working/edit_2d.log"

os.makedirs(BACKUP, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

env = os.environ.copy()
env.update({
    "PATH": f"{VENV}/bin:" + env.get("PATH", ""),
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "2",
    "NCCL_P2P_DISABLE": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "FREEFINE_ATTN_CHUNK": "8",          # fp32, attention NOT chunked
})

s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
cmd = [f"{VENV}/bin/torchrun", "--nproc_per_node=2", f"--master-port={port}", SCRIPT]

print(f"[start] Cell 8 — fp32 edit step, 2 ranks  log={LOG}", flush=True)
log_f = open(LOG, "w")
proc = subprocess.Popen(
    cmd, cwd="/kaggle/temp/GeoBenchMeta", env=env,
    stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid,
)
print(f"[start] pid={proc.pid}", flush=True)

def count_done():
    if not os.path.isdir(OUTDIR): return 0
    n = 0
    for _, _, files in os.walk(OUTDIR):
        n += sum(1 for f in files if f.endswith('.png'))
    return n

def backup_outputs():
    if os.path.isdir(OUTDIR):
        subprocess.run(["rsync", "-a", OUTDIR + "/", BACKUP + "/"],
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

start = time.time(); last_report = 0.0; last_backup = 0.0
REPORT_EVERY = 30; BACKUP_EVERY = 600

try:
    while proc.poll() is None:
        time.sleep(2); now = time.time()
        if now - last_report >= REPORT_EVERY:
            count = count_done(); ts = datetime.datetime.now().strftime("%H:%M:%S")
            elapsed_min = (now - start) / 60
            if count > 0 and elapsed_min > 0:
                rate = count / elapsed_min
                print(f"[{ts}] generated={count}  elapsed={elapsed_min:.1f}min  rate={rate:.1f}/min", flush=True)
            else:
                print(f"[{ts}] generated={count}  (loading / first cases)", flush=True)
            last_report = now
        if now - last_backup >= BACKUP_EVERY:
            backup_outputs()
            print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] backup → {BACKUP}", flush=True)
            last_backup = now
except KeyboardInterrupt:
    print("[interrupt] killing pgid", flush=True)
    try: os.killpg(os.getpgid(proc.pid), signal.SIGINT); proc.wait(timeout=30)
    except Exception:
        try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception: pass

backup_outputs()
log_f.close()
count = count_done()
print(f"[done] rc={proc.returncode}  generated={count}/5677  log={LOG}", flush=True)

[start] Cell 8 — fp32 edit step, 2 ranks  log=/kaggle/working/edit_2d.log
[start] pid=16247
[21:52:19] generated=3680  elapsed=0.0min  rate=110363.6/min
[21:52:19] backup → /kaggle/working/gen_results_2d_backup
[21:52:49] generated=3680  elapsed=0.5min  rate=6852.5/min
[21:53:19] generated=3680  elapsed=1.0min  rate=3545.8/min
[21:53:49] generated=3680  elapsed=1.5min  rate=2391.7/min
[21:54:19] generated=3682  elapsed=2.0min  rate=1805.4/min
[21:54:49] generated=3682  elapsed=2.5min  rate=1449.5/min
[21:55:19] generated=3684  elapsed=3.0min  rate=1211.4/min
[21:55:49] generated=3686  elapsed=3.5min  rate=1040.7/min
[21:56:19] generated=3688  elapsed=4.0min  rate=912.3/min
[21:56:49] generated=3689  elapsed=4.5min  rate=812.0/min
[21:57:19] generated=3690  elapsed=5.0min  rate=731.6/min
[21:57:49] generated=3692  elapsed=5.5min  rate=665.8/min
[21:58:20] generated=3694  elapsed=6.0min  rate=611.0/min
[21:58:50] generated=3695  elapsed=6.5min  rate=564.4/min
[21:59:20] generated=3697  e

 sync + free space + tar:

In [7]:
%%bash
set -e
rsync -a /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/ \
         /kaggle/working/gen_results_2d_backup/
echo "backup count (must be 5677):"
find /kaggle/working/gen_results_2d_backup -name '*.png' | wc -l

rm -rf /kaggle/working/freefine_env
rm -f  /kaggle/working/gen_results_2d_partial.tar
df -h /kaggle/working

TAR=/kaggle/working/gen_results_2d_final.tar    # renamed to mark it as the full set
tar -cf $TAR -C /kaggle/working gen_results_2d_backup
ls -lh $TAR
df -h /kaggle/working

backup count (must be 5677):
5677
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  9.0G   11G  46% /kaggle/working
-rw-r--r-- 1 root root 2.2G May 24 08:54 /kaggle/working/gen_results_2d_final.tar
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   12G  8.5G  57% /kaggle/working


Cell 9 — set up the evaluation env (separate venv, different torch)


New session!!

Step 0 — verify what survived persistence

In [1]:
%%bash
echo "=== /kaggle/working state ==="
ls -lh /kaggle/working/*.tar 2>/dev/null
echo
echo "=== FreeFine repo + ckpts? ==="
ls /kaggle/working/FreeFine/checkpoints/sd-15/unet/ 2>/dev/null | head -3
grep -c "ori_mask = read_and_resize_mask" /kaggle/working/FreeFine/evaluation/FreeFine/freefine_batch_infer_2d.py
echo
echo "=== free disk ==="
df -h /kaggle/working

=== /kaggle/working state ===
-rw-r--r-- 1 root root 2.2G May 24 10:28 /kaggle/working/gen_results_2d_final.tar
-rw-r--r-- 1 root root 220M May 24 10:27 /kaggle/working/inp_img_blended.tar

=== FreeFine repo + ckpts? ===
config.json
diffusion_pytorch_model.bin
1

=== free disk ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   12G  8.5G  57% /kaggle/working


Step 1 — set HF_TOKEN

In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN set, length:", len(os.environ["HF_TOKEN"]))

HF_TOKEN set, length: 37



Step 2 — re-download GeoBenchMeta 2D subset (~15 min)

In [3]:
import os
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="CIawevy/GeoBenchMeta",
    repo_type="dataset",
    local_dir="/kaggle/temp/GeoBenchMeta",
    token=os.environ["HF_TOKEN"],
    max_workers=2,
    etag_timeout=60,
    allow_patterns=["annotation_2d.json", "Geo-Bench-2D/**"],
)
!du -sh /kaggle/temp/GeoBenchMeta && ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

Fetching ... files: 0it [00:00, ?it/s]

3.0G	/kaggle/temp/GeoBenchMeta
coarse_img    source_img	  source_mask
inp_mask_vis  source_img_full_v2  target_mask


Step 3 — symlink annotations + restore both tars

In [4]:
%%bash
set -e
# Symlink for the misnamed annotations file
ln -sf /kaggle/temp/GeoBenchMeta/annotation_2d.json /kaggle/temp/GeoBenchMeta/annotations_2d.json

# Restore Cell 7's 668 bg-gen outputs into the live data tree
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D
tar -xf /kaggle/working/inp_img_blended.tar -C /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

# Restore the FINAL Cell 8 outputs (5677 PNGs)
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d
tar -xf /kaggle/working/gen_results_2d_final.tar -C /kaggle/working
rsync -a /kaggle/working/gen_results_2d_backup/ \
         /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/

echo "counts (must be 668 and 5677):"
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended -name 'inp_img.png' | wc -l
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' | wc -l

counts (must be 668 and 5677):
668
5677


Step 4 — rebuild the venv (~5 min)

In [5]:
%%bash
set -e

pip install -q --root-user-action=ignore uv
uv python install 3.10.13

VENV=/kaggle/working/freefine_env
PY=$VENV/bin/python
rm -rf "$VENV"
uv venv --python 3.10.13 "$VENV"
"$PY" -c "import sys; print('venv:', sys.version.split()[0], sys.executable)"

uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" \
    --index-url https://download.pytorch.org/whl/cu121

cd /kaggle/working/FreeFine
uv pip install --python "$PY" -r requirements.txt || {
  echo "Retrying with xformers unpinned..."
  grep -v '^xformers' requirements.txt > /tmp/req_noxf.txt
  uv pip install --python "$PY" -r /tmp/req_noxf.txt
  uv pip install --python "$PY" xformers
}

uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70" wheel pip

"$PY" - <<'PY'
import sys, torch, diffusers, xformers, transformers, pkg_resources, pytorch_lightning
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)
print("diffusers", diffusers.__version__)
print("xformers", xformers.__version__)
print("transformers", transformers.__version__)
print("pytorch_lightning", pytorch_lightning.__version__)
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 69.5 MB/s eta 0:00:00
venv: 3.10.13 /kaggle/working/freefine_env/bin/python
python 3.10.13
torch 2.1.1+cu121 cuda True 12.1
diffusers 0.18.0
xformers 0.0.23
transformers 4.30.1
pytorch_lightning 2.1.3


Installed Python 3.10.13 in 2.19s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: freefine_env
Activate with: source freefine_env/bin/activate
Using Python 3.10.13 environment at: freefine_env
Resolved 18 packages in 1.36s
Prepared 18 packages in 28.42s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 5.05s
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/working/freefine_env
Re

Replacement for Cell 9 — uv-based metric_env +cleanup

In [8]:
%%bash
set -e

# freefine_env served its purpose in the last session — Cell 8 is done.
# Metrics live in their own venv.
rm -rf /kaggle/working/freefine_env

# Wipe the half-built metric_env from the failed install.
rm -rf /kaggle/working/metric_env

df -h /kaggle/working

# Rebuild metric_env via uv.
uv venv --python 3.10.13 /kaggle/working/metric_env
PY=/kaggle/working/metric_env/bin/python
"$PY" -c "import sys; print('metric_env python:', sys.version.split()[0])"

uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124

uv pip install --python "$PY" -r /kaggle/working/FreeFine/evaluation/metrics/requirements.txt

CLIP_DIR=$(find /kaggle/working/metric_env -type d -name clip 2>/dev/null | head -1)
if [ -n "$CLIP_DIR" ] && [ ! -f "$CLIP_DIR/bpe_simple_vocab_16e6.txt.gz" ]; then
    echo "fetching CLIP BPE vocab into $CLIP_DIR"
    wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P "$CLIP_DIR"
fi

"$PY" -c "import torch; print('metric_env torch:', torch.__version__, 'cuda:', torch.cuda.is_available())"
df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   12G  8.5G  57% /kaggle/working
metric_env python: 3.10.13


Using CPython 3.10.13
Creating virtual environment at: metric_env
Activate with: source metric_env/bin/activate
Using Python 3.10.13 environment at: metric_env
Resolved 27 packages in 730ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 27 packages in 5.54s
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4.127
 + nvidia-cuda-nvrtc-cu12==12.4.127
 + nvidia-cuda-runtime-cu12==12.4.127
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.2.1.3
 + nvidia-curand-cu12==10.3.5.147
 + nvidia-cusolver-cu12==11.6.1.9
 + nvidia-cusparse-cu12==12.3.1.170
 + nvidia-cusparselt-cu12==0.6.2
 + nvidia-nccl-cu12==2.21.5
 + nvidia-nvjitlink-cu12==12.4.127
 + nvidia-nvtx-cu12=

CalledProcessError: Command 'b'set -e\n\n# freefine_env served its purpose in the last session \xe2\x80\x94 Cell 8 is done.\n# Metrics live in their own venv.\nrm -rf /kaggle/working/freefine_env\n\n# Wipe the half-built metric_env from the failed install.\nrm -rf /kaggle/working/metric_env\n\ndf -h /kaggle/working\n\n# Rebuild metric_env via uv.\nuv venv --python 3.10.13 /kaggle/working/metric_env\nPY=/kaggle/working/metric_env/bin/python\n"$PY" -c "import sys; print(\'metric_env python:\', sys.version.split()[0])"\n\nuv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \\\n    --index-url https://download.pytorch.org/whl/cu124\n\nuv pip install --python "$PY" -r /kaggle/working/FreeFine/evaluation/metrics/requirements.txt\n\nCLIP_DIR=$(find /kaggle/working/metric_env -type d -name clip 2>/dev/null | head -1)\nif [ -n "$CLIP_DIR" ] && [ ! -f "$CLIP_DIR/bpe_simple_vocab_16e6.txt.gz" ]; then\n    echo "fetching CLIP BPE vocab into $CLIP_DIR"\n    wget -q http

In [9]:
%%bash
set -e
PY=/kaggle/working/metric_env/bin/python

# 1) Ensure setuptools (with pkg_resources) is in the env for any build step that needs it.
uv pip install --python "$PY" "setuptools<70" wheel

# 2) Install everything in metrics/requirements.txt EXCEPT clip.
grep -v "openai/CLIP" /kaggle/working/FreeFine/evaluation/metrics/requirements.txt > /tmp/req_noclip.txt
uv pip install --python "$PY" -r /tmp/req_noclip.txt

# 3) Install clip from the exact pinned commit, without build isolation
#    so it picks up the venv's setuptools.
uv pip install --python "$PY" --no-build-isolation \
    "git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"

# 4) CLIP BPE vocab — fetch if needed.
CLIP_DIR=$(find /kaggle/working/metric_env -type d -name clip 2>/dev/null | head -1)
if [ -n "$CLIP_DIR" ] && [ ! -f "$CLIP_DIR/bpe_simple_vocab_16e6.txt.gz" ]; then
    echo "fetching CLIP BPE vocab into $CLIP_DIR"
    wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P "$CLIP_DIR"
fi

# 5) Sanity check
"$PY" -c "import torch, clip; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), '| clip ok')"
df -h /kaggle/working

torch: 2.6.0+cu124 cuda: True | clip ok
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   17G  2.9G  86% /kaggle/working


Using Python 3.10.13 environment at: metric_env
Resolved 3 packages in 158ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 3 packages in 83ms
 + packaging==26.2
 + setuptools==69.5.1
 + wheel==0.47.0
Using Python 3.10.13 environment at: metric_env
Resolved 94 packages in 2.22s
Prepared 35 packages in 2.77s
Uninstalled 1 package in 37ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 68 packages in 3.51s
 + accelerate==1.3.0
 + aiohappyeyeballs==2.6.2
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + args==0.1.0
 + async-timeout==5.0.1
 + attrs==26.1.0
 + braceexpand==0.1.7
 + certifi==2026.5.20
 + charset-normalizer==3.4.7
 

Cell 10 — run the 2D metrics
(For per-difficulty splits, repeat with --level 1, --level 2, --level 3)


Step 10.0 — verify the JSON exists, reconstruct if not

In [10]:
import os, json, glob

GEO   = "/kaggle/temp/GeoBenchMeta"
GEN   = f"{GEO}/Geo-Bench-2D/Gen_results_FreeFine_2d"
JSON_FILE = f"{GEO}/generated_results_freefine_2d.json"

if os.path.isfile(JSON_FILE):
    with open(JSON_FILE) as f:
        data = json.load(f)
    n = sum(len(v.get("instances", {})) for v in data.values()) if isinstance(data, dict) else len(data)
    print(f"JSON present — top-level entries: {len(data) if isinstance(data, dict) else 'list'}, instance count: {n}")
else:
    print("JSON missing — will reconstruct from on-disk PNGs + annotation_2d.json")
    
    # Inspect actual filename convention first
    samples = glob.glob(f"{GEN}/*/*/*/*.png")[:5]
    print("sample generated paths (first 5):")
    for s in samples: print(" ", s)
    print(f"total PNGs in Gen_results_FreeFine_2d: {len(glob.glob(f'{GEN}/*/*/*/*.png'))}")
    
    with open(f"{GEO}/annotation_2d.json") as f:
        ann = json.load(f)
    
    print(f"annotation_2d.json top-level entries (da_n): {len(ann)}")
    # Probe one entry's shape
    da_keys = list(ann.keys())
    print(f"first da_n key: {da_keys[0]}")
    print(f"  shape under da_n: {list(ann[da_keys[0]].keys())}")
    if "instances" in ann[da_keys[0]]:
        ins_keys = list(ann[da_keys[0]]["instances"].keys())
        print(f"  instances under first da_n: {ins_keys[:3]}...")
        ins0 = ann[da_keys[0]]["instances"][ins_keys[0]]
        print(f"  shape under first instance: {list(ins0.keys())}")
        if "edits" in ins0:
            edits_keys = list(ins0["edits"].keys())
            print(f"  edits under first instance: {edits_keys}")
            print(f"  first edit content keys: {list(ins0['edits'][edits_keys[0]].keys())}")

JSON missing — will reconstruct from on-disk PNGs + annotation_2d.json
sample generated paths (first 5):
total PNGs in Gen_results_FreeFine_2d: 0
annotation_2d.json top-level entries (da_n): 609
first da_n key: 0
  shape under da_n: ['4v_caption', 'instances']
  instances under first da_n: ['0']...
  shape under first instance: ['0', '1', '2', '3', '4', '5', '6', '7', '8']


In [11]:
%%bash
echo "=== tars in /kaggle/working ==="
ls -lh /kaggle/working/*.tar 2>/dev/null

echo
echo "=== /kaggle/temp/GeoBenchMeta/Geo-Bench-2D subdirs ==="
ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/

echo
echo "=== inp_img_blended count (should be 668) ==="
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended -name '*.png' 2>/dev/null | wc -l

echo
echo "=== Gen_results_FreeFine_2d count ==="
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' 2>/dev/null | wc -l

echo
echo "=== gen_results_2d_backup count (intermediate from tar extract) ==="
find /kaggle/working/gen_results_2d_backup -name '*.png' 2>/dev/null | wc -l

echo
echo "=== free disk ==="
df -h /kaggle/working

=== tars in /kaggle/working ===
-rw-r--r-- 1 root root 2.2G May 24 10:28 /kaggle/working/gen_results_2d_final.tar
-rw-r--r-- 1 root root 220M May 24 10:27 /kaggle/working/inp_img_blended.tar

=== /kaggle/temp/GeoBenchMeta/Geo-Bench-2D subdirs ===
coarse_img
Gen_results_FreeFine_2d
inp_img_blended
inp_mask_vis
source_img
source_img_full_v2
source_mask
target_mask

=== inp_img_blended count (should be 668) ===
668

=== Gen_results_FreeFine_2d count ===
5677

=== gen_results_2d_backup count (intermediate from tar extract) ===
5677

=== free disk ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   17G  2.9G  86% /kaggle/working


In [12]:
%%bash
echo "=== sample PNG paths ==="
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' | head -10
echo
echo "=== depth distribution (number of '/' in each path) ==="
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' | awk -F/ '{print NF}' | sort | uniq -c
echo
echo "=== top-level entries ==="
ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d | head -5
echo "(... total: $(ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d | wc -l))"

=== sample PNG paths ===
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/8.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/2.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/4.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/6.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/5.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/3.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/0.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/7.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/260/0/1.png
/kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/199/0/0.png

=== depth distribution (number of '/' in each path) ===
   5677 9

=== top-level entries ===
0
1
10
100
101
(... total: 609)


Reconstruction cell


In [13]:
import os, json

GEO       = "/kaggle/temp/GeoBenchMeta"
GEN       = f"{GEO}/Geo-Bench-2D/Gen_results_FreeFine_2d"
OUT_JSON  = f"{GEO}/generated_results_freefine_2d.json"

with open(f"{GEO}/annotation_2d.json") as f:
    ann = json.load(f)

added = missing = total = 0
for da_n, da in ann.items():
    for ins_id, current_ins in da.get("instances", {}).items():
        for edit_ins, input_pack in current_ins.items():
            total += 1
            png_abs = os.path.join(GEN, da_n, ins_id, f"{edit_ins}.png")
            if os.path.isfile(png_abs):
                # Relative to base_dir so --use_relative_path + --base_dir works.
                input_pack["gen_img_path"] = os.path.relpath(png_abs, GEO)
                added += 1
            else:
                missing += 1

with open(OUT_JSON, "w") as f:
    json.dump(ann, f)

print(f"total edits in annotation: {total}")
print(f"added gen_img_path:        {added}")
print(f"missing (no PNG on disk):  {missing}")
print(f"wrote {OUT_JSON}  ({os.path.getsize(OUT_JSON)/1024:.0f} KB)")

total edits in annotation: 5677
added gen_img_path:        5677
missing (no PNG on disk):  0
wrote /kaggle/temp/GeoBenchMeta/generated_results_freefine_2d.json  (3149 KB)


Pre-run cleanup + redirect HF cache


In [14]:
%%bash
# Drop the redundant rsync'd copy (we have the live data in /kaggle/temp/ + the tar).
rm -rf /kaggle/working/gen_results_2d_backup
df -h /kaggle/working
df -h /kaggle/temp || df -h /

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  5.0G  75% /kaggle/working
Filesystem      Size  Used Avail Use% Mounted on
overlay         7.9T  6.7T  1.2T  86% /


Cell 10 — metrics run


Patch — semantically equivalent


In [16]:
import pathlib, re

p = pathlib.Path("/kaggle/working/FreeFine/evaluation/metrics/main.py")
src = p.read_text()

patched, n = re.subn(
    r"if\s+args\.3d:\s*\n(\s+)data = parse_data_3d\(data\)",
    r"if getattr(args, '3d'):\n\1data = parse_data_3d(data)",
    src,
)
assert n == 1, f"expected 1 replacement, got {n}"
p.write_text(patched)
print(f"patched main.py — replaced {n} occurrence of `args.3d` → `getattr(args, '3d')`")

patched main.py — replaced 1 occurrence of `args.3d` → `getattr(args, '3d')`


In [18]:
%%bash
uv pip install --python /kaggle/working/metric_env/bin/python "pyarrow<16"
/kaggle/working/metric_env/bin/python -c "from datasets import load_dataset; import ImageReward as RM; print('datasets + ImageReward import ok')"

datasets + ImageReward import ok


Using Python 3.10.13 environment at: metric_env
Resolved 2 packages in 178ms
Prepared 1 package in 621ms
Uninstalled 2 packages in 85ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 1.05s
 - numpy==2.2.2
 + numpy==1.26.4
 - pyarrow==24.0.0
 + pyarrow==15.0.2


In [20]:
%%bash
set -e
HPS_DIR=/kaggle/working/metric_env/lib/python3.10/site-packages/hpsv2/src/open_clip
CLIP_DIR=$(find /kaggle/working/metric_env -path '*/site-packages/clip' -type d 2>/dev/null | head -1)
if [ -n "$CLIP_DIR" ] && [ -f "$CLIP_DIR/bpe_simple_vocab_16e6.txt.gz" ]; then
    cp "$CLIP_DIR/bpe_simple_vocab_16e6.txt.gz" "$HPS_DIR/"
    echo "copied from $CLIP_DIR"
else
    wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P "$HPS_DIR"
    echo "downloaded fresh"
fi
ls -lh "$HPS_DIR/bpe_simple_vocab_16e6.txt.gz"

copied from /kaggle/working/metric_env/lib/python3.10/site-packages/clip
-rw-r--r-- 1 root root 1.3M May 24 11:22 /kaggle/working/metric_env/lib/python3.10/site-packages/hpsv2/src/open_clip/bpe_simple_vocab_16e6.txt.gz


Patch dift_sd.py to use the mirror - The original stabilityai/stable-diffusion-2-1 was taken down


In [23]:
import pathlib
p = pathlib.Path("/kaggle/working/FreeFine/evaluation/metrics/MD/dift_sd.py")
src = p.read_text()

old = "'stabilityai/stable-diffusion-2-1'"
new = "'sd2-community/stable-diffusion-2-1'"
assert old in src, "couldn't find the SD-2.1 reference in dift_sd.py"
p.write_text(src.replace(old, new))
print("patched: SD-2.1 hub id → sd2-community mirror")

patched: SD-2.1 hub id → sd2-community mirror


Patch mean_distance.py


In [25]:
import pathlib
p = pathlib.Path("/kaggle/working/FreeFine/evaluation/metrics/MD/mean_distance.py")
src = p.read_text()

old = "SDFeaturizer('stabilityai/stable-diffusion-2-1')"
new = "SDFeaturizer('sd2-community/stable-diffusion-2-1')"
assert old in src, "couldn't find the SDFeaturizer call site"
p.write_text(src.replace(old, new))
print("patched mean_distance.py:114 — SD-2.1 hub id → sd2-community mirror")

patched mean_distance.py:114 — SD-2.1 hub id → sd2-community mirror


In [26]:
%%bash
set -e
source /kaggle/working/metric_env/bin/activate

# Redirect all HuggingFace / torch.hub / clip caches to /kaggle/temp (~1 TB free, session-scoped).
export MPLBACKEND=Agg
export HF_HOME=/kaggle/temp/.cache/huggingface
export TRANSFORMERS_CACHE=/kaggle/temp/.cache/huggingface
export HF_DATASETS_CACHE=/kaggle/temp/.cache/huggingface
export TORCH_HOME=/kaggle/temp/.cache/torch
export CLIP_HOME=/kaggle/temp/.cache/clip
mkdir -p $HF_HOME $TORCH_HOME $CLIP_HOME

cd /kaggle/working/FreeFine/evaluation/metrics
python main.py \
  --path /kaggle/temp/GeoBenchMeta/generated_results_freefine_2d.json \
  --use_relative_path \
  --base_dir /kaggle/temp/GeoBenchMeta \
  --fid_path /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/source_img_full_v2 \
  --task 100111111 \
  --level 0 \
  2>&1 | tee /kaggle/working/metrics_2d.log

/kaggle/working/metric_env/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
-----FID-----
100%|██████████| 89/89 [00:35<00:00,  2.47it/s]
FID: 35.04393467706461
-----Background Consistency-----
100%|██████████| 5677/5677 [05:19<00:00, 17.75it/s]
Using cache found in /kaggle/temp/.cache/torch/hub/facebookresearch_dino_main
BGC: 0.9670339197639598
-----Subject Consistency-----
100%|██████████| 5677/5677 [05:09<00:00, 18.32it/s]
SUBC: 0.9113285210490836
-----Wrap Error-----
100%|██████████| 609/609 [02:26<00:00,  4.15it/s]
WRAP_E: 0.04737241250867297
-----MD-----
Fetching 11 files: 100%|██████████| 11/11 [00:16<00:00,  1.54s/it]
Expected types for unet: ['UNet2DConditionModel'], got MyUNet2DConditionModel.
  0%|          | 1/5677 [00:13<21:05:17, 13.38s/it]
Traceback (most recent call last):
  File "/kaggle/working/FreeFine/evaluation/metric

New session

Step 1 — verify what survived persistence


In [ ]:
%%bash
echo "=== tars in /kaggle/working ==="
ls -lh /kaggle/working/*.tar 2>/dev/null
echo
echo "=== FreeFine repo + patches survived? ==="
grep -c "getattr(args, '3d')" /kaggle/working/FreeFine/evaluation/metrics/main.py
grep -c "sd2-community/stable-diffusion-2-1" /kaggle/working/FreeFine/evaluation/metrics/MD/mean_distance.py
grep -c "sd2-community/stable-diffusion-2-1" /kaggle/working/FreeFine/evaluation/metrics/MD/dift_sd.py
echo
df -h /kaggle/working

Step 2 — HF_TOKEN


In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN set, length:", len(os.environ["HF_TOKEN"]))

env: HF_TOKEN=[REDACTED_EXPIRED_HF_TOKEN]
HF_TOKEN set, length: 37


Step 3 — re-download GeoBenchMeta (~15 min)


In [3]:
import os
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="CIawevy/GeoBenchMeta",
    repo_type="dataset",
    local_dir="/kaggle/temp/GeoBenchMeta",
    token=os.environ["HF_TOKEN"],
    max_workers=2,
    etag_timeout=60,
    allow_patterns=["annotation_2d.json", "Geo-Bench-2D/**"],
)
!du -sh /kaggle/temp/GeoBenchMeta && ls /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

Fetching ... files: 0it [00:00, ?it/s]

3.0G	/kaggle/temp/GeoBenchMeta
coarse_img    source_img	  source_mask
inp_mask_vis  source_img_full_v2  target_mask


Step 4 — symlinks + restore tars + verify counts


In [4]:
%%bash
set -e
ln -sf /kaggle/temp/GeoBenchMeta/annotation_2d.json /kaggle/temp/GeoBenchMeta/annotations_2d.json
mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D
tar -xf /kaggle/working/inp_img_blended.tar -C /kaggle/temp/GeoBenchMeta/Geo-Bench-2D

mkdir -p /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d
tar -xf /kaggle/working/gen_results_2d_final.tar -C /kaggle/working
rsync -a /kaggle/working/gen_results_2d_backup/ \
         /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d/

echo "counts (must be 668 and 5677):"
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/inp_img_blended -name 'inp_img.png' | wc -l
find /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/Gen_results_FreeFine_2d -name '*.png' | wc -l

counts (must be 668 and 5677):
668
5677


Step 5 — rebuild the JSON manifest


In [5]:
import os, json

GEO       = "/kaggle/temp/GeoBenchMeta"
GEN       = f"{GEO}/Geo-Bench-2D/Gen_results_FreeFine_2d"
OUT_JSON  = f"{GEO}/generated_results_freefine_2d.json"

with open(f"{GEO}/annotation_2d.json") as f:
    ann = json.load(f)

added = missing = total = 0
for da_n, da in ann.items():
    for ins_id, current_ins in da.get("instances", {}).items():
        for edit_ins, input_pack in current_ins.items():
            total += 1
            png_abs = os.path.join(GEN, da_n, ins_id, f"{edit_ins}.png")
            if os.path.isfile(png_abs):
                input_pack["gen_img_path"] = os.path.relpath(png_abs, GEO)
                added += 1
            else:
                missing += 1
with open(OUT_JSON, "w") as f:
    json.dump(ann, f)
print(f"total={total} added={added} missing={missing}")

total=5677 added=5677 missing=0


Step 6 — clean disk + rebuild metric_env (all fixes baked in)


In [18]:
%%bash
set -e
# Frees space if leftover from prior session
rm -rf /kaggle/working/freefine_env
rm -rf /kaggle/working/metric_env
rm -rf /kaggle/working/gen_results_2d_backup
df -h /kaggle/working

# uv install + Python 3.10
pip install -q --root-user-action=ignore uv
uv python install 3.10.13

# Build the metric venv
uv venv --python 3.10.13 /kaggle/working/metric_env
PY=/kaggle/working/metric_env/bin/python

# setuptools<70 first so pkg_resources is available for clip's build
uv pip install --python "$PY" "setuptools<70" wheel

# torch 2.6 + cu124
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124

# Requirements minus clip
grep -v "openai/CLIP" /kaggle/working/FreeFine/evaluation/metrics/requirements.txt > /tmp/req_noclip.txt
uv pip install --python "$PY" -r /tmp/req_noclip.txt

# clip from the same pinned commit, no build isolation so it uses our setuptools
uv pip install --python "$PY" --no-build-isolation \
    "git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"

# Pin pyarrow so `datasets` doesn't break on the removed PyExtensionType
uv pip install --python "$PY" "pyarrow<16"

# Place the CLIP BPE vocab in both locations that need it
CLIP_DIR=$(find /kaggle/working/metric_env -path '*/site-packages/clip' -type d | head -1)
HPS_DIR=/kaggle/working/metric_env/lib/python3.10/site-packages/hpsv2/src/open_clip
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P "$CLIP_DIR"
cp "$CLIP_DIR/bpe_simple_vocab_16e6.txt.gz" "$HPS_DIR/"

"$PY" -c "import torch, clip; from datasets import load_dataset; import ImageReward; print('imports ok | torch', torch.__version__, 'cuda', torch.cuda.is_available())"
df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  9.0G   11G  46% /kaggle/working
imports ok | torch 2.6.0+cu124 cuda True
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  5.0G  75% /kaggle/working


Python 3.10.13 is already installed
Using CPython 3.10.13
Creating virtual environment at: metric_env
Activate with: source metric_env/bin/activate
Using Python 3.10.13 environment at: metric_env
Resolved 3 packages in 108ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 3 packages in 38ms
 + packaging==26.2
 + setuptools==69.5.1
 + wheel==0.47.0
Using Python 3.10.13 environment at: metric_env
Resolved 27 packages in 598ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 27 packages in 3.15s
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6

Step 7 — Run MD only (~25 min)


In [ ]:
%%bash
set -e
source /kaggle/working/metric_env/bin/activate

export MPLBACKEND=Agg
export HF_HOME=/kaggle/temp/.cache/huggingface
export HF_DATASETS_CACHE=/kaggle/temp/.cache/huggingface
export TORCH_HOME=/kaggle/temp/.cache/torch
export CLIP_HOME=/kaggle/temp/.cache/clip
export HF_TOKEN="$HF_TOKEN"
export HUGGING_FACE_HUB_TOKEN="$HF_TOKEN"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
mkdir -p $HF_HOME $TORCH_HOME $CLIP_HOME

cd /kaggle/working/FreeFine/evaluation/metrics
python main.py \
  --path /kaggle/temp/GeoBenchMeta/generated_results_freefine_2d.json \
  --use_relative_path \
  --base_dir /kaggle/temp/GeoBenchMeta \
  --fid_path /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/source_img_full_v2 \
  --task 000000100 \
  --level 0 \
  2>&1 | tee -a /kaggle/working/metrics_2d.log

Cell 11 — same for the 3D refinement scripts


In [ ]:
%%bash
set -e
cd /kaggle/working/FreeFine/evaluation/FreeFine
export NCCL_P2P_DISABLE=1
FREE_PORT=$(python -c 'import socket; s=socket.socket(); s.bind(("", 0)); print(s.getsockname()[1]); s.close()')
torchrun --nproc_per_node=2 --master-port $FREE_PORT freefine_batch_infer_bggen_3d.py
torchrun --nproc_per_node=2 --master-port $FREE_PORT freefine_batch_infer_3d_depth.py